# c04 - non-sequential vs sequential ray tracing: noise / time / memory

What this measures, in one line: **how many rays a differentiable ray trace needs to hold
per-bin Monte-Carlo noise under 10 %, and what that costs in seconds and GPU bytes.**

Per-bin relative noise is `eps = 1 / sqrt(rays per lit bin)`, and rays per lit bin is
`k * R / N**2` for a receiver of `N x N` bins and `R` launched rays. So bin count is
**not a free axis** - the sweep raises `R` and raises `N` together to pin `eps` just under
10 %, then reports time and memory at each rung. That is what maps the practical limit.

Two arms:

| arm | tracer | coating |
|---|---|---|
| `seq_R00` | dO `Lensgroup.trace` | R = 0 (all a sequential tracer can do) |
| `nonseq_mc_R02` | `diffoptics.nonseq.trace_mc` | R = 0.2, Russian-roulette MC |

**The comparison is compound.** The two arms differ in *both* tracer and scene - dO has no
partial reflection, so `seq @ R=0.2` does not exist. Every ratio below mixes the cost of
non-sequentiality with the cost of branching. `--arm nonseq_mc_R00` splits it into two clean
ratios if you want them.

Everything runs in **fp64**. On a T4 that is the 1/32-rate path, so expect roughly laptop-CPU
speed - that is a *finding*, not a bug, and the `--device cpu` cell puts it in the CSV as data.

**Runtime:** Runtime > Change runtime type > **T4 GPU**. Budget ~1 h for a full sweep.
Every cell is safe to re-run verbatim after a disconnect - results are append-only and
completed runs are skipped.

In [ ]:
# --- environment -------------------------------------------------------------
!nvidia-smi

import torch, platform
print('torch      ', torch.__version__)
print('cuda avail ', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('device     ', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f'VRAM       {total/2**30:.1f} GiB total, {free/2**30:.1f} GiB free')
print('python     ', platform.python_version())
print()
print('NOTE fp64 runs at 1/32 the fp32 rate on T4/L4, 1/2 on A100/V100.')
print('     On a 1/32-rate part expect roughly CPU speed.')

## 0. Get the code onto the runtime

The notebook only *drives* `c04_bench.py`; it does not contain it. The runtime needs the
`diffoptics` package plus `examples/nonseq/{c02_R02.py, c04_bench.py}`. Run **A or B**, not both.

**A - git clone** (preferred: reproducible, records a commit hash in `results.json`).
**B - upload `c04_colab_bundle.zip`** (64 KB, in `examples/nonseq/`) - no fork or PAT needed,
but `git_head` in the metadata will read `unknown`.

In [ ]:
# --- OPTION A: clone the branch ---------------------------------------------
# Put a GitHub PAT in Colab Secrets under the name GH_PAT (key icon, left sidebar).
# Never paste a token into a cell - it gets saved inside the notebook.
from google.colab import userdata

REPO   = 'YOURUSER/DiffOptics-nonseq'   # <-- your fork
BRANCH = 'bench'

import os, subprocess
if not os.path.isdir('/content/DiffOptics'):
    tok = userdata.get('GH_PAT')
    url = f'https://{tok}@github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    url, '/content/DiffOptics'], check=True)
%cd /content/DiffOptics
!git log --oneline -1
!pip install -q matplotlib psutil

In [ ]:
# --- OPTION B: upload the bundle instead of cloning --------------------------
# Pick examples/nonseq/c04_colab_bundle.zip when the file chooser opens.
import os, zipfile
if not os.path.isdir('/content/DiffOptics'):
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('/content')
%cd /content/DiffOptics
!ls diffoptics examples/nonseq
!pip install -q matplotlib psutil

In [ ]:
# --- outputs go to Drive, not the container ---------------------------------
# A disconnect then costs at most the single run that was in flight.
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/diffoptics_bench/c04_out'
import os; os.makedirs(OUT, exist_ok=True)
BENCH = '/content/DiffOptics/examples/nonseq/c04_bench.py'
print(OUT)

## 1. Smoke - under 60 s

Does every code path execute, and does the energy ledger close? Run this before committing
an hour of GPU. It writes into a *separate* directory so it never pollutes the real CSV.

In [ ]:
!python $BENCH --smoke --device cuda --out $OUT/../smoke

## 2. Calibrate `k`

One pilot trace per arm, splatted at N in {16 ... 512}. Two things to eyeball:

* `slope_f` must be **-2.00 +- 0.10** - count per bin falls as `N**-2`, or the geometry is wrong.
* `k_mc / k_seq` must be **0.64** - the `T1T2` path fraction at R = 0.2, since at R = 0 every
  captured ray transmits. A free cross-check on the whole path-fraction story.

In [ ]:
!python $BENCH --calibrate-only --device cuda --out $OUT

## 3. Sequential arm - ~3 min

Cheap arm first, so the CSV is never empty if the session dies.

In [ ]:
!python $BENCH --arm seq_R00 --device cuda --out $OUT

## 4. Non-sequential MC arm - the long one

**Re-run this cell verbatim after a disconnect.** Completed runs are keyed on
`(arm, device, rays, seed, rep, chunk, n_bins)` and skipped; the sweep resumes where it stopped.

Runs ascend in ray count, so an interrupted session still leaves a complete low-R curve with
error bars rather than one unfinished 1e8. Rows above 1e7 switch to chunked accumulation -
those carry `mem_mode='chunked'` and their peak memory is **not** comparable with the
monolithic rows (the plots keep them on separate lines).

In [ ]:
!python $BENCH --arm nonseq_mc_R02 --device cuda --out $OUT

## 5. Optional extras

* **fixed-N slope check** - the sharpest single test in the harness: at fixed `N`, `eps` must
  fall as `R**-1/2`. Validates the noise measurement, the splat, the classifier and RNG
  independence at once. Gate: **-0.500 +- 0.02**, fitted on `eps_median` (the p10 statistic is
  the right headline for sizing bins but is itself biased at small integer counts).
* **CPU ladder** - the fp64 finding. Slow, so keep it to R <= 1e6.
* **`nonseq_mc_R00`** - decomposes the compound comparison.
* **`split_R02`** - deterministic branching tracer; expect 5-8x the bytes per ray, which is
  what turns "MC is the memory-scalable one" from assertion into measurement.

In [ ]:
!python $BENCH --slope --device cuda --seeds 2 --reps 1 --out $OUT
# !python $BENCH --device cpu --seeds 1 --reps 1 --rays 1e4 3e4 1e5 3e5 1e6 --out $OUT
# !python $BENCH --arm nonseq_mc_R00 --device cuda --out $OUT
# !python $BENCH --arm split_R02 --device cuda --rays 1e4 1e5 1e6 --out $OUT

## 6. Plots and summary

In [ ]:
!python $BENCH --plots-only --out $OUT
!python $BENCH --json-only  --out $OUT

from IPython.display import Image, display
for f in ('noise_vs_rays.png', 'time_vs_rays.png', 'memory_vs_rays.png'):
    display(Image(filename=f'{OUT}/{f}'))

In [ ]:
import pandas as pd, json
df = pd.read_csv(f'{OUT}/results.csv')
print(df.status.value_counts().to_dict())

ok = df[(df.status == 'ok') & (df['mode'] == 'adaptive')]
cols = ['n_bins_fwd', 'n_bins_back', 'eps_p10_fwd', 'frac_above_fwd',
        't_wall_s', 'mem_alloc_mb']
display(ok.groupby(['arm', 'rays'])[cols].mean().round(4))

print(json.dumps(json.load(open(f'{OUT}/results.json'))['arms'], indent=2)[:2000])

## 7. Findings - fill these in from the run above

1. **Cost per ray.** `us_per_ray` and `bytes_per_ray` per arm, straight out of `results.json`.
   Quote the raw non-seq/seq ratio *and* a per-surface-interaction-normalised one: the raw
   number compares 2 fixed refractions against up to 10 closest-hit rounds over 2 elements,
   so the honest overhead lies between the two.
2. **The OOM wall.** Largest R that fitted, smallest that failed, and the wall predicted from
   the linear bytes-per-ray fit. Predicted ~ observed is what validates linearity - `trace_mc`
   is flat-`[N]`-width by construction, so bytes per ray must not depend on depth.
3. **Bins at 10 % noise.** `N` reached at each `R`, forward vs backward. The forward/backward
   gap at equal budget quantifies how much harder the reflected path is to resolve - a
   statement no sequential tracer can make at all.
4. **fp64 on a T4.** GPU vs CPU wall time at matched R. If they are close, say so plainly:
   the differentiable non-sequential tracer in fp64 runs no faster on a T4 than on a laptop CPU.
5. **The compound-comparison caveat**, restated in the report itself. The sequential arm also
   cannot produce `R1`, `T1R2T1` or the ghost at all - `n_bins_back` is NaN on those rows by
   construction, not by omission.
6. **Floored rows.** Low-R rows marked `bin_flag='floored'` sit above 10 % because `N` hit its
   16-bin floor. They are kept deliberately: they anchor the `-1/2` slope check. Do not read
   them as failures.